In [1]:
# =====================================================================
# QUESTION 6.2
# McNEMAR'S TEST
# Optimised Logistic Regression vs Strongest Comparator (Deep MLP)
# =====================================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix
)

from scipy.stats import binomtest


SEED = 42
ALPHA = 0.05


# =====================================================================
# 1. LOAD DATASET
# =====================================================================

PATH = r"C:\Users\Sekomane\Desktop\Data ScienceTUT\NIWIS_Water Supply Reliability - Population_26-Sep-2026.csv"

df = pd.read_csv(PATH)


print("=" * 85)
print("QUESTION 6.2 - McNEMAR'S TEST")
print("=" * 85)

print("Dataset shape:", df.shape)


# =====================================================================
# 2. CLEAN DATA
# =====================================================================

data = df.copy()


# Remove unnamed columns
data = data.loc[
    :,
    ~data.columns.str.contains("^Unnamed")
]


# Standardise spaces in column names
data.columns = (
    data.columns
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
)


def clean_number(series):

    return pd.to_numeric(
        series.astype(str)
              .str.replace(" ", "", regex=False)
              .str.replace(",", "", regex=False)
              .str.strip(),
        errors="coerce"
    )


# Convert population/count columns
for col in data.columns:

    if "%" not in col and col != "WSA Name":

        data[col] = clean_number(
            data[col]
        )


# Convert percentage columns
for col in data.columns:

    if "%" in col:

        data[col] = pd.to_numeric(
            data[col]
                .astype(str)
                .str.replace("%", "", regex=False)
                .str.strip(),
            errors="coerce"
        )


# =====================================================================
# 3. CREATE TARGET
# =====================================================================

# Class 0 = Reliable
# Class 1 = Low Reliability

data["low_reliability"] = (
    data["% Population Reliable Supply"] < 60
).astype(int)


# =====================================================================
# 4. REMOVE LEAKAGE VARIABLES
# =====================================================================

leakage_columns = [

    "Population Reliable Supply",
    "% Population Reliable Supply",

    "Urban Population Reliable Supply",
    "% Urban Population Reliable Supply",

    "Rural Population Reliable Supply",
    "% Rural Population Reliable Supply"
]


y = data["low_reliability"]


X = data.drop(
    columns=[
        "low_reliability",
        "WSA Name"
    ] + leakage_columns,
    errors="ignore"
).copy()


# =====================================================================
# 5. FEATURE ENGINEERING
# =====================================================================

X["population_access_rate"] = (
    X["Population with access to water"]
    / X["Population"]
) * 100


X["urban_access_rate"] = (
    X["Urban Population with access to water"]
    / X["Urban Population"]
) * 100


X["rural_access_rate"] = (
    X["Rural Population with access to water"]
    / X["Rural Population"]
) * 100


X["urban_population_share"] = (
    X["Urban Population"]
    / X["Population"]
) * 100


X["rural_population_share"] = (
    X["Rural Population"]
    / X["Population"]
) * 100


X = X.replace(
    [np.inf, -np.inf],
    np.nan
)


# =====================================================================
# 6. RECREATE EXACT SAME HELD-OUT SPLIT
# =====================================================================

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)


print("\nDevelopment observations:", len(X_dev))
print("Held-out observations    :", len(X_test))


# =====================================================================
# 7. OPTIMISED CHAMPION
# =====================================================================

# Locked Logistic Regression configuration from Q5.

champion_model = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "scaler",
        RobustScaler()
    ),

    (
        "classifier",
        LogisticRegression(
            C=1,
            penalty="l1",
            solver="liblinear",
            class_weight=None,
            max_iter=5000,
            random_state=SEED
        )
    )
])


# =====================================================================
# 8. STRONGEST COMPARATOR - DEEP MLP
# =====================================================================

# Same Deep MLP configuration used in Question 4.

comparator_model = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "scaler",
        RobustScaler()
    ),

    (
        "classifier",
        MLPClassifier(
            hidden_layer_sizes=(64, 32, 16),
            activation="relu",
            solver="adam",
            alpha=0.005,
            batch_size=16,
            learning_rate_init=0.001,
            max_iter=1200,
            early_stopping=True,
            validation_fraction=0.20,
            n_iter_no_change=30,
            random_state=SEED
        )
    )
])


# =====================================================================
# 9. TRAIN BOTH MODELS ON SAME DEVELOPMENT DATA
# =====================================================================

champion_model.fit(
    X_dev,
    y_dev
)


comparator_model.fit(
    X_dev,
    y_dev
)


# =====================================================================
# 10. PREDICT EXACT SAME HELD-OUT CASES
# =====================================================================

champion_pred = champion_model.predict(
    X_test
)


comparator_pred = comparator_model.predict(
    X_test
)


# =====================================================================
# 11. PERFORMANCE CHECK
# =====================================================================

champion_accuracy = accuracy_score(
    y_test,
    champion_pred
)


champion_f1 = f1_score(
    y_test,
    champion_pred,
    average="macro"
)


comparator_accuracy = accuracy_score(
    y_test,
    comparator_pred
)


comparator_f1 = f1_score(
    y_test,
    comparator_pred,
    average="macro"
)


print("\n" + "=" * 85)
print("HELD-OUT PERFORMANCE")
print("=" * 85)


print("\nOptimised Logistic Regression")

print(
    "Accuracy:",
    round(champion_accuracy, 3)
)

print(
    "Macro-F1:",
    round(champion_f1, 3)
)


print("\nDeep MLP")

print(
    "Accuracy:",
    round(comparator_accuracy, 3)
)

print(
    "Macro-F1:",
    round(comparator_f1, 3)
)


# =====================================================================
# 12. DETERMINE WHETHER EACH PREDICTION IS CORRECT
# =====================================================================

champion_correct = (
    champion_pred == y_test.to_numpy()
)


comparator_correct = (
    comparator_pred == y_test.to_numpy()
)


# =====================================================================
# 13. BUILD McNEMAR 2 x 2 CORRECTNESS TABLE
# =====================================================================

# a = both models correct
# b = champion correct, comparator wrong
# c = champion wrong, comparator correct
# d = both models wrong

a = np.sum(
    champion_correct
    & comparator_correct
)


b = np.sum(
    champion_correct
    & ~comparator_correct
)


c = np.sum(
    ~champion_correct
    & comparator_correct
)


d = np.sum(
    ~champion_correct
    & ~comparator_correct
)


mcnemar_table = pd.DataFrame(

    [
        [a, b],
        [c, d]
    ],

    index=[
        "Champion Correct",
        "Champion Wrong"
    ],

    columns=[
        "Comparator Correct",
        "Comparator Wrong"
    ]
)


print("\n" + "=" * 85)
print("2 x 2 CORRECTNESS TABLE")
print("=" * 85)


display(
    mcnemar_table
)


print("\nBoth correct:", a)

print(
    "Champion correct / Comparator wrong:",
    b
)

print(
    "Champion wrong / Comparator correct:",
    c
)

print(
    "Both wrong:",
    d
)


print(
    "\nTotal:",
    a + b + c + d
)


# =====================================================================
# 14. HYPOTHESES
# =====================================================================

print("\n" + "=" * 85)
print("McNEMAR HYPOTHESES")
print("=" * 85)


print(
    "\nH0: The optimised Logistic Regression and Deep MLP "
    "have equal probabilities of making an error on the "
    "same held-out cases."
)


print(
    "\nH1: The two models have different probabilities "
    "of making an error on the same held-out cases."
)


print(
    "\nSignificance level (alpha):",
    ALPHA
)


# =====================================================================
# 15. EXACT McNEMAR TEST
# =====================================================================

# McNemar's exact test uses ONLY the discordant pairs:
#
# b = champion correct, comparator wrong
# c = champion wrong, comparator correct
#
# Under H0, each discordant prediction is equally likely
# to favour either model (p = 0.5).


discordant = b + c


print("\n" + "=" * 85)
print("EXACT McNEMAR TEST")
print("=" * 85)


print(
    "\nDiscordant pairs:",
    discordant
)


if discordant == 0:

    p_value = 1.0
    exact_statistic = 0

    print(
        "There are no discordant predictions."
    )

else:

    # Exact binomial version of McNemar's test
    exact_result = binomtest(
        k=min(b, c),
        n=discordant,
        p=0.5,
        alternative="two-sided"
    )

    p_value = exact_result.pvalue

    # For the exact version, the smaller discordant count
    # can be reported as the test statistic.
    exact_statistic = min(
        b,
        c
    )


print(
    "\nExact statistic:",
    exact_statistic
)


print(
    "Exact p-value:",
    round(
        p_value,
        6
    )
)


# =====================================================================
# 16. STATISTICAL DECISION
# =====================================================================

print("\n" + "=" * 85)
print("STATISTICAL DECISION")
print("=" * 85)


if p_value < ALPHA:

    print(
        "\np-value < 0.05"
    )

    print(
        "Decision: Reject H0."
    )

    print(
        "There is statistically significant evidence "
        "that the two models differ in their error rates."
    )

else:

    print(
        "\np-value >= 0.05"
    )

    print(
        "Decision: Fail to reject H0."
    )

    print(
        "There is insufficient statistical evidence "
        "that the two models differ in their error rates."
    )


# =====================================================================
# 17. PRACTICAL DIFFERENCE
# =====================================================================

accuracy_difference = (
    champion_accuracy
    - comparator_accuracy
)


macro_f1_difference = (
    champion_f1
    - comparator_f1
)


print("\n" + "=" * 85)
print("PRACTICAL PERFORMANCE DIFFERENCE")
print("=" * 85)


print(
    "\nAccuracy difference "
    "(Champion - Comparator):",
    round(
        accuracy_difference,
        3
    )
)


print(
    "Macro-F1 difference "
    "(Champion - Comparator):",
    round(
        macro_f1_difference,
        3
    )
)


print(
    "\nStatistical significance and practical significance "
    "are not the same. McNemar's test evaluates whether the "
    "models differ significantly in paired correctness, while "
    "the metric differences describe the practical size and "
    "direction of the performance difference."
)


# =====================================================================
# 18. CASE-LEVEL EVIDENCE
# =====================================================================

comparison_cases = pd.DataFrame({

    "Actual":
        y_test.to_numpy(),

    "Optimised Logistic Prediction":
        champion_pred,

    "Deep MLP Prediction":
        comparator_pred,

    "Optimised Logistic Correct":
        champion_correct,

    "Deep MLP Correct":
        comparator_correct
})


print("\n" + "=" * 85)
print("SAME HELD-OUT CASES - PREDICTION COMPARISON")
print("=" * 85)


display(
    comparison_cases
)


# =====================================================================
# 19. COMPLETION
# =====================================================================

print("\n" + "=" * 85)

print(
    "QUESTION 6.2 COMPLETED SUCCESSFULLY"
)

print("=" * 85)

ModuleNotFoundError: No module named 'pandas'